Determine how often true entity matches involve:
    - same-script representations
    - cross-script representations
    - non-ASCII representations
    - missing addresses

This experiment is intended to inform:
    1. Candidate generation strategy
    2. Transliteration requirements
    3. Multilingual embedding model selection
    4. Source-specific matching strategies

Outputs
-------
experiments/E001_5/
    source_script_distribution.csv
    true_pair_script_analysis.csv
    true_pair_script_summary.csv
    cross_script_examples.csv
    checkpoint.txt
    progress.log


In [3]:
def validate_ground_truth_schema():

    sample = pd.read_csv(
        GT_PATH,
        sep="\t",
        dtype=str,
        nrows=5,
        keep_default_na=False,
    )

    expected = {
        "source1_entity_id",
        "matched_entity_ids",
    }

    actual = set(sample.columns)

    if actual != expected:

        raise ValueError(
            "Unexpected ground-truth schema.\n"
            f"Expected: {sorted(expected)}\n"
            f"Found:    {sorted(actual)}"
        )

    log(
        "Ground-truth schema validated: "
        "source1_entity_id, matched_entity_ids"
    )

In [5]:
"""
E001.5 - Script / Representation Analysis

The true-pair analysis is streamed. It does NOT accumulate all
7.6M true pairs in Python memory.
"""

from __future__ import annotations

import csv
import os
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

BASE_DIR = Path(r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource")

DATA_DIR = BASE_DIR / "dataset"
OUTPUT_DIR = BASE_DIR /"amazon-er" / "outputs" / "E001.5_outputs"

S1_PATH = DATA_DIR / "train"/"train_source1.tsv"
S2_PATH = DATA_DIR / "train"/"train_source2.tsv"
S3_PATH = DATA_DIR / "train"/"train_source3.tsv"
GT_PATH = DATA_DIR / "train"/"train_ground_truth.tsv"

CHUNK_SIZE = 100_000

# Number of examples to save for each cross-script category
MAX_EXAMPLES_PER_CATEGORY = 100

# Resume from checkpoint
RESUME = True


# ============================================================
# SETUP
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DISTRIBUTION_PATH = OUTPUT_DIR / "source_script_distribution.csv"
PAIR_ANALYSIS_PATH = OUTPUT_DIR / "true_pair_script_analysis.csv"
SUMMARY_PATH = OUTPUT_DIR / "true_pair_script_summary.csv"
EXAMPLES_PATH = OUTPUT_DIR / "cross_script_examples.csv"
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.txt"
LOG_PATH = OUTPUT_DIR / "progress.log"


# ============================================================
# LOGGING
# ============================================================

def log(message: str):
    """Print and persist progress information."""

    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

    line = f"[{timestamp}] {message}"

    print(line, flush=True)

    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


# ============================================================
# UNICODE SCRIPT DETECTION
# ============================================================

SCRIPT_RANGES = {
    "Latin": [
        (0x0041, 0x005A),
        (0x0061, 0x007A),
        (0x00C0, 0x024F),
        (0x1E00, 0x1EFF),
    ],

    "Devanagari": [
        (0x0900, 0x097F),
        (0xA8E0, 0xA8FF),
    ],

    "Bengali": [
        (0x0980, 0x09FF),
    ],

    "Gurmukhi": [
        (0x0A00, 0x0A7F),
    ],

    "Gujarati": [
        (0x0A80, 0x0AFF),
    ],

    "Oriya": [
        (0x0B00, 0x0B7F),
    ],

    "Tamil": [
        (0x0B80, 0x0BFF),
    ],

    "Telugu": [
        (0x0C00, 0x0C7F),
    ],

    "Kannada": [
        (0x0C80, 0x0CFF),
    ],

    "Malayalam": [
        (0x0D00, 0x0D7F),
    ],

    "Sinhala": [
        (0x0D80, 0x0DFF),
    ],

    "Arabic": [
        (0x0600, 0x06FF),
        (0x0750, 0x077F),
        (0x08A0, 0x08FF),
    ],

    "Cyrillic": [
        (0x0400, 0x04FF),
        (0x0500, 0x052F),
    ],

    "Greek": [
        (0x0370, 0x03FF),
    ],

    "Hebrew": [
        (0x0590, 0x05FF),
    ],

    "Thai": [
        (0x0E00, 0x0E7F),
    ],

    "Georgian": [
        (0x10A0, 0x10FF),
        (0x2D00, 0x2D2F),
    ],

    "Armenian": [
        (0x0530, 0x058F),
    ],
}


def get_script(char: str) -> str:
    """Return the Unicode script for a character."""

    cp = ord(char)

    for script, ranges in SCRIPT_RANGES.items():
        for start, end in ranges:
            if start <= cp <= end:
                return script

    return "Other"


def detect_scripts(text) -> str:
    """
    Detect scripts represented in a string.

    Returns:
        Latin
        Devanagari
        Latin+Devanagari
        Other
        Empty
    """

    if pd.isna(text):
        return "Missing"

    text = str(text).strip()

    if not text:
        return "Empty"

    scripts = set()

    for char in text:
        category = unicodedata.category(char)

        # Ignore numbers, punctuation, spaces, symbols, etc.
        if not category.startswith("L"):
            continue

        scripts.add(get_script(char))

    if not scripts:
        return "NoLetters"

    return "+".join(sorted(scripts))


def has_non_ascii(text) -> bool:

    if pd.isna(text):
        return False

    return any(ord(c) > 127 for c in str(text))


# ============================================================
# HELPERS
# ============================================================

def classify_script_relationship(script_a: str, script_b: str) -> str:

    if script_a == "Missing" or script_b == "Missing":
        return "missing"

    if script_a == script_b:
        return "same"

    # Exact set comparison is useful when one side is mixed.
    set_a = set(script_a.split("+"))
    set_b = set(script_b.split("+"))

    if set_a & set_b:
        return "overlap"

    return "cross_script"


def initialize_csv(path: Path, columns: list[str]):

    if path.exists():
        return

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(columns)


def write_rows(path: Path, rows: list[list]):

    if not rows:
        return

    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerows(rows)


# ============================================================
# PHASE 1
# SOURCE SCRIPT DISTRIBUTION
# ============================================================

def analyze_source(source_name: str, path: Path):

    log(f"Starting source analysis: {source_name}")
    log(f"File: {path}")

    counters = {
        "name_script": {},
        "address_script": {},
        "name_non_ascii": 0,
        "address_non_ascii": 0,
        "rows": 0,
    }

    start = time.time()

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False,
        )
    ):

        for value in chunk["business_name"]:
            script = detect_scripts(value)
            counters["name_script"][script] = (
                counters["name_script"].get(script, 0) + 1
            )

            if has_non_ascii(value):
                counters["name_non_ascii"] += 1

        for value in chunk["business_address"]:
            script = detect_scripts(value)
            counters["address_script"][script] = (
                counters["address_script"].get(script, 0) + 1
            )

            if has_non_ascii(value):
                counters["address_non_ascii"] += 1

        counters["rows"] += len(chunk)

        if chunk_no % 5 == 0:

            elapsed = time.time() - start

            rate = counters["rows"] / elapsed if elapsed else 0

            log(
                f"{source_name}: "
                f"{counters['rows']:,} rows | "
                f"{rate:,.0f} rows/s | "
                f"elapsed {elapsed/60:.1f} min"
            )

    rows = []

    for field in ["name_script", "address_script"]:

        total = counters["rows"]

        for script, count in sorted(
            counters[field].items(),
            key=lambda x: -x[1],
        ):

            rows.append([
                source_name,
                field.replace("_script", ""),
                script,
                count,
                count / total if total else 0,
            ])

    rows.extend([
        [
            source_name,
            "name",
            "NON_ASCII",
            counters["name_non_ascii"],
            counters["name_non_ascii"] / counters["rows"],
        ],
        [
            source_name,
            "address",
            "NON_ASCII",
            counters["address_non_ascii"],
            counters["address_non_ascii"] / counters["rows"],
        ],
    ])

    return rows


def run_source_distribution():

    log("=" * 70)
    log("PHASE 1: SOURCE SCRIPT DISTRIBUTION")
    log("=" * 70)

    initialize_csv(
        DISTRIBUTION_PATH,
        [
            "source",
            "field",
            "script",
            "count",
            "fraction",
        ],
    )

    # Avoid duplicating output if the phase was already completed.
    if DISTRIBUTION_PATH.exists() and DISTRIBUTION_PATH.stat().st_size > 100:
        log("Source distribution already exists; skipping Phase 1.")
        return

    for source_name, path in [
        ("S1", S1_PATH),
        ("S2", S2_PATH),
        ("S3", S3_PATH),
    ]:

        rows = analyze_source(source_name, path)
        write_rows(DISTRIBUTION_PATH, rows)

        log(f"Completed {source_name}")


# ============================================================
# PHASE 2
# BUILD COMPACT SCRIPT LOOKUPS
# ============================================================

def build_script_lookup(source_name: str, path: Path):

    """
    Build a compact lookup:

        entity_id -> {
            name_script,
            address_script,
            name_non_ascii,
            address_non_ascii
        }

    This is intentionally separate from the original E001 feature
    lookup. We only keep the information needed for this experiment.
    """

    log(f"Building script lookup for {source_name}")

    lookup = {}

    start = time.time()
    rows_processed = 0

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            path,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False,
            usecols=[
                "entity_id",
                "business_name",
                "business_address",
            ],
        )
    ):

        for row in chunk.itertuples(index=False):

            lookup[row.entity_id] = (
                detect_scripts(row.business_name),
                detect_scripts(row.business_address),
                has_non_ascii(row.business_name),
                has_non_ascii(row.business_address),
                row.business_name,
                row.business_address,
            )

        rows_processed += len(chunk)

        if chunk_no % 5 == 0:

            elapsed = time.time() - start
            rate = rows_processed / elapsed if elapsed else 0

            log(
                f"{source_name} lookup: "
                f"{rows_processed:,} rows | "
                f"{rate:,.0f} rows/s"
            )

    log(
        f"{source_name} lookup complete: "
        f"{len(lookup):,} records"
    )

    return lookup


# ============================================================
# PHASE 3
# TRUE-PAIR ANALYSIS
# ============================================================

def load_checkpoint():

    if not RESUME or not CHECKPOINT_PATH.exists():
        return 0

    try:
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    except Exception:
        return 0


def save_checkpoint(chunk_number: int):

    temp_path = CHECKPOINT_PATH.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        f.write(str(chunk_number))

    os.replace(temp_path, CHECKPOINT_PATH)


def run_true_pair_analysis():

    log("=" * 70)
    log("PHASE 2: TRUE-PAIR SCRIPT ANALYSIS")
    log("=" * 70)
    
    validate_ground_truth_schema()

    s1_lookup = build_script_lookup("S1", S1_PATH)
    s2_lookup = build_script_lookup("S2", S2_PATH)
    s3_lookup = build_script_lookup("S3", S3_PATH)

    initialize_csv(
        PAIR_ANALYSIS_PATH,
        [
            "s1_id",
            "target_id",
            "target_source",

            "s1_name_script",
            "target_name_script",

            "s1_address_script",
            "target_address_script",

            "name_relationship",
            "address_relationship",

            "s1_name_non_ascii",
            "target_name_non_ascii",

            "s1_address_non_ascii",
            "target_address_non_ascii",

            "s1_name",
            "target_name",
        ],
    )

    initialize_csv(
        EXAMPLES_PATH,
        [
            "s1_id",
            "target_id",
            "target_source",
            "s1_name",
            "target_name",
            "s1_name_script",
            "target_name_script",
            "s1_address_script",
            "target_address_script",
            "name_relationship",
            "address_relationship",
        ],
    )

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    stats = {
        "total_pairs": 0,

        "s2_pairs": 0,
        "s3_pairs": 0,

        "name_same": 0,
        "name_overlap": 0,
        "name_cross_script": 0,
        "name_missing": 0,

        "address_same": 0,
        "address_overlap": 0,
        "address_cross_script": 0,
        "address_missing": 0,

        "name_non_ascii_pairs": 0,
        "address_non_ascii_pairs": 0,

        "both_name_cross_address_cross": 0,
        "name_cross_only": 0,
        "address_cross_only": 0,
    }

    example_counts = {}

    start_chunk = load_checkpoint()

    if start_chunk:
        log(
            f"RESUMING from GT chunk {start_chunk:,}"
        )

    start_time = time.time()
    rows_processed = 0

    for chunk_no, gt in enumerate(
        pd.read_csv(
            GT_PATH,
            sep="\t",
            dtype=str,
            chunksize=CHUNK_SIZE,
            keep_default_na=False,
        )
    ):

        if chunk_no < start_chunk:
            continue

        output_rows = []
        example_rows = []

        for row in gt.itertuples(index=False):

            # Actual challenge GT schema:
            #
            # source1_entity_id
            # matched_entity_ids
            #
            # matched_entity_ids is a comma-separated list
            # containing S2-* and S3-* entity IDs.
            s1_id = str(row.source1_entity_id).strip()

            matched_value = str(row.matched_entity_ids).strip()
            if not matched_value:
                target_ids = []
            else:
                target_ids = [target_id.strip()for target_id in matched_value.split(",")if target_id.strip()]

            s1_info = s1_lookup.get(s1_id)

            if s1_info is None:
                continue

            (
                s1_name_script,
                s1_address_script,
                s1_name_non_ascii,
                s1_address_non_ascii,
                s1_name,
                s1_address,
            ) = s1_info

            for target_id in target_ids:

                if target_id.startswith("S2"):
                    target_info = s2_lookup.get(target_id)
                    target_source = "S2"

                elif target_id.startswith("S3"):
                    target_info = s3_lookup.get(target_id)
                    target_source = "S3"

                else:
                    # Fallback: try both
                    target_info = s2_lookup.get(target_id)

                    if target_info is not None:
                        target_source = "S2"
                    else:
                        target_info = s3_lookup.get(target_id)
                        target_source = "S3"

                if target_info is None:
                    continue

                (
                    target_name_script,
                    target_address_script,
                    target_name_non_ascii,
                    target_address_non_ascii,
                    target_name,
                    target_address,
                ) = target_info

                name_rel = classify_script_relationship(
                    s1_name_script,
                    target_name_script,
                )

                address_rel = classify_script_relationship(
                    s1_address_script,
                    target_address_script,
                )

                stats["total_pairs"] += 1

                if target_source == "S2":
                    stats["s2_pairs"] += 1
                else:
                    stats["s3_pairs"] += 1

                stats[f"name_{name_rel}"] += 1
                stats[f"address_{address_rel}"] += 1

                if s1_name_non_ascii or target_name_non_ascii:
                    stats["name_non_ascii_pairs"] += 1

                if s1_address_non_ascii or target_address_non_ascii:
                    stats["address_non_ascii_pairs"] += 1

                name_cross = name_rel == "cross_script"
                address_cross = address_rel == "cross_script"

                if name_cross and address_cross:
                    stats["both_name_cross_address_cross"] += 1

                elif name_cross:
                    stats["name_cross_only"] += 1

                elif address_cross:
                    stats["address_cross_only"] += 1

                output_rows.append([
                    s1_id,
                    target_id,
                    target_source,

                    s1_name_script,
                    target_name_script,

                    s1_address_script,
                    target_address_script,

                    name_rel,
                    address_rel,

                    s1_name_non_ascii,
                    target_name_non_ascii,

                    s1_address_non_ascii,
                    target_address_non_ascii,

                    s1_name,
                    target_name,
                ])

                # Save a limited number of examples for each
                # interesting relationship.
                categories = []

                if name_cross:
                    categories.append("name_cross_script")

                if address_cross:
                    categories.append("address_cross_script")

                if name_cross and address_cross:
                    categories.append("both_cross_script")

                for category in categories:

                    current = example_counts.get(category, 0)

                    if current < MAX_EXAMPLES_PER_CATEGORY:

                        example_rows.append([
                            s1_id,
                            target_id,
                            target_source,
                            s1_name,
                            target_name,
                            s1_name_script,
                            target_name_script,
                            s1_address_script,
                            target_address_script,
                            name_rel,
                            address_rel,
                        ])

                        example_counts[category] = current + 1

        # ----------------------------------------------------
        # WRITE OUTPUT IMMEDIATELY
        # ----------------------------------------------------

        write_rows(PAIR_ANALYSIS_PATH, output_rows)
        write_rows(EXAMPLES_PATH, example_rows)

        save_checkpoint(chunk_no + 1)

        rows_processed += len(gt)

        elapsed = time.time() - start_time
        rate = rows_processed / elapsed if elapsed else 0

        # Total GT rows are known from EDA, but we don't hard-code
        # that assumption into processing.
        log(
            f"GT chunk {chunk_no:,} complete | "
            f"{rows_processed:,} S1 records processed | "
            f"{stats['total_pairs']:,} true pairs | "
            f"{rate:,.0f} S1/s | "
            f"elapsed {elapsed/60:.1f} min"
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    write_summary(stats)

    log("TRUE-PAIR ANALYSIS COMPLETE")


# ============================================================
# SUMMARY
# ============================================================

def write_summary(stats):

    total = stats["total_pairs"]

    rows = []

    def add(metric, count):

        rows.append([
            metric,
            count,
            count / total if total else 0,
        ])

    add("total_true_pairs", total)

    add("S2_true_pairs", stats["s2_pairs"])
    add("S3_true_pairs", stats["s3_pairs"])

    add("name_same_script", stats["name_same"])
    add("name_script_overlap", stats["name_overlap"])
    add("name_cross_script", stats["name_cross_script"])
    add("name_missing", stats["name_missing"])

    add("address_same_script", stats["address_same"])
    add("address_script_overlap", stats["address_overlap"])
    add("address_cross_script", stats["address_cross_script"])
    add("address_missing", stats["address_missing"])

    add(
        "pairs_with_non_ascii_name",
        stats["name_non_ascii_pairs"],
    )

    add(
        "pairs_with_non_ascii_address",
        stats["address_non_ascii_pairs"],
    )

    add(
        "both_name_and_address_cross_script",
        stats["both_name_cross_address_cross"],
    )

    add(
        "name_cross_script_only",
        stats["name_cross_only"],
    )

    add(
        "address_cross_script_only",
        stats["address_cross_only"],
    )

    pd.DataFrame(
        rows,
        columns=[
            "metric",
            "count",
            "fraction_of_true_pairs",
        ],
    ).to_csv(
        SUMMARY_PATH,
        index=False,
    )


# ============================================================
# MAIN
# ============================================================

def main():

    total_start = time.time()

    log("=" * 70)
    log("E001.5 SCRIPT / REPRESENTATION ANALYSIS")
    log("=" * 70)

    log(f"S1: {S1_PATH}")
    log(f"S2: {S2_PATH}")
    log(f"S3: {S3_PATH}")
    log(f"GT: {GT_PATH}")

    run_source_distribution()

    run_true_pair_analysis()

    elapsed = time.time() - total_start

    log("=" * 70)
    log(
        f"E001.5 COMPLETE | "
        f"total runtime: {elapsed/60:.2f} minutes"
    )
    log("=" * 70)


if __name__ == "__main__":
    main()

[2026-09-26 13:14:25] ======================================================================
[2026-09-26 13:14:25] E001.5 SCRIPT / REPRESENTATION ANALYSIS
[2026-09-26 13:14:25] ======================================================================
[2026-09-26 13:14:25] S1: E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_source1.tsv
[2026-09-26 13:14:25] S2: E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_source2.tsv
[2026-09-26 13:14:25] S3: E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_source3.tsv
[2026-09-26 13:14:25] GT: E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_ground_truth.tsv
[2026-09-26 13:14:25] ======================================================================
[2026-09-26 13:14:25] PHASE 1: SOURCE SCRIPT DISTRIBUTION
[2026-09-26 13:14:25] ======================================================================
[2026-09-26 13:14:25] Source distribution already exist

[2026-09-26 13:14:25] Ground-truth schema validated: source1_entity_id, matched_entity_ids
[2026-09-26 13:14:25] Building script lookup for S1
[2026-09-26 13:14:28] S1 lookup: 100,000 rows | 39,813 rows/s
[2026-09-26 13:14:40] S1 lookup: 600,000 rows | 39,623 rows/s
[2026-09-26 13:14:55] S1 lookup: 1,100,000 rows | 36,699 rows/s
[2026-09-26 13:15:10] S1 lookup: 1,600,000 rows | 35,714 rows/s
[2026-09-26 13:15:25] S1 lookup: 2,100,000 rows | 34,855 rows/s
[2026-09-26 13:15:29] S1 lookup complete: 2,206,821 records
[2026-09-26 13:15:29] Building script lookup for S2
[2026-09-26 13:15:32] S2 lookup: 100,000 rows | 35,297 rows/s
[2026-09-26 13:15:44] S2 lookup: 600,000 rows | 39,134 rows/s
[2026-09-26 13:15:56] S2 lookup: 1,100,000 rows | 41,036 rows/s
[2026-09-26 13:16:08] S2 lookup: 1,600,000 rows | 41,199 rows/s
[2026-09-26 13:16:19] S2 lookup: 2,100,000 rows | 41,476 rows/s
[2026-09-26 13:16:31] S2 lookup: 2,600,000 rows | 41,716 rows/s
[2026-09-26 13:16:43] S2 lookup: 3,100,000 rows |

In [2]:
gt_sample = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
    nrows=5,
    keep_default_na=False,
)

print("Columns:")
print(gt_sample.columns.tolist())

print("\nShape:")
print(gt_sample.shape)

print("\nSample:")
display(gt_sample)

Columns:
['source1_entity_id', 'matched_entity_ids']

Shape:
(5, 2)

Sample:


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
